In [3]:
import torch

In [2]:
# CrossEntropyLoss 範例
# 用和 D1 NumPy 題相同的數字：logits = [[2.0, 1.0, 0.1]]、正確類別 = 0。
# 下面把 PyTorch 算出來的 loss、梯度，和手算公式並排印出來對照。

# logits：模型最後一層的原始分數，形狀 (N, C)。
# N 是 batch 筆數，C 是類別數。這裡 N=1、C=3，三個分數依序是類別 0、1、2。
# CrossEntropyLoss 吃的是 logits，不是 softmax 之後的機率。
# requires_grad=True：backward 之後才會把 d(loss)/d(logits) 寫進 logits.grad。
logits = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)

# target：每一筆的正確類別，形狀 (N,)，值是 0 到 C-1 的整數索引。
# 這裡唯一這筆的答案是類別 0，對應 logits 裡的 2.0。
# 傳進去的是類別編號，不是 one-hot 向量。
target = torch.tensor([0])

# 預設 reduction="mean"：先算每一筆的 loss，再對這 N 筆取平均。
# 內部是 log_softmax 再加負號，等價於：
#   loss_i = -log_softmax(logits_i)[target_i]
#   loss   = mean_i(loss_i)
# log_softmax 會先減掉該列最大值再取 log，logits 到 ±1000 時仍然是有限數。
# 先 softmax 再 log，遇到很大的 logits 會先溢位成 inf。
CEL = torch.nn.CrossEntropyLoss()  # reduction="mean"
loss = CEL(logits, target)

# backward 從這個純量 loss 往回傳。
# reduction="mean" 時，logits 的梯度是：
#   dlogits = (softmax(logits) - one_hot(target)) / N
# 正確類別那一格是負的，其餘類別是正的；每一列加起來是 0。
# reduction="sum" 時不除以 N，梯度會變成 mean 的 N 倍。
loss.backward()

# detach：下面只是拿數值來對照公式，不再接進計算圖，也不會改到剛才寫好的 logits.grad。
probs = torch.softmax(logits.detach(), dim=-1)

# one_hot 和 logits 同形狀 (N, C)。正確類別的位置是 1，其餘是 0。
# 這個例子只有第 0 列，所以列索引寫 0；target 指到該列的正確類別。
one_hot = torch.zeros_like(probs)
one_hot[0, target] = 1

# 手算 loss：每一列用 [0, 1, ..., N-1] 配上 target，挑出正確類別的 log_softmax。
# 加負號後對 N 筆取平均，和 CrossEntropyLoss 的 mean 是同一條路徑。
manual_loss = -torch.log_softmax(logits.detach(), dim=-1)[range(len(target)), target].mean()

# 手算梯度。mean 要除以 N；這裡 N=1，除不除結果相同。
# 印出來應該和 logits.grad 一致：大約 [-0.341, 0.242, 0.099]。
manual_grad = (probs - one_hot) / logits.shape[0]  # mean 時除以 N

print("loss", loss.detach().item(), manual_loss.item())
print("grad", logits.grad)
print("softmax - one_hot / N", manual_grad)

loss 0.4170299470424652 0.4170299470424652
grad tensor([[-0.3410,  0.2424,  0.0986]])
softmax - one_hot / N tensor([[-0.3410,  0.2424,  0.0986]])


In [3]:
# 真的更新參數：backward 只寫 .grad，step 才改數值
# 還是同一組數字。正確類別是 0，所以更新後 logits[0] 應變大，另外兩個應變小。

logits = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
target = torch.tensor([0])
lr = 1.0  # 故意設大，一步就能看出差值

# 把 logits 登記成「要被更新的參數」。沒有這行，opt.step() 不會動任何東西。
opt = torch.optim.SGD([logits], lr=lr)
CEL = torch.nn.CrossEntropyLoss()

print("更新前 logits", logits.detach().clone())

loss = CEL(logits, target)
loss.backward()  # 只寫 logits.grad，logits 數值還沒變
print("grad        ", logits.grad)
print("backward 後 ", logits.detach().clone())  # 仍是 [[2.0, 1.0, 0.1]]

# SGD：param = param - lr * grad
# 等價手寫（不要和 step 同時跑，會更新兩次）：
# with torch.no_grad():
#     logits -= lr * logits.grad
opt.step()
print("step 後     ", logits.detach().clone())

opt.zero_grad()  # 清掉 .grad，否則下一次 backward 會把新梯度累加進去
print("zero_grad 後", logits.grad)

更新前 logits tensor([[2.0000, 1.0000, 0.1000]])
grad         tensor([[-0.3410,  0.2424,  0.0986]])
backward 後  tensor([[2.0000, 1.0000, 0.1000]])
step 後      tensor([[2.3410e+00, 7.5757e-01, 1.4341e-03]])
zero_grad 後 None


In [4]:
# MLP 完整訓練步驟
# 結構和 D1 選配題相同：x → Linear → ReLU → Linear → CrossEntropyLoss
# 這次被更新的是 Linear 的 weight/bias，不是 logits 本身。
# 一步順序：zero_grad → forward → loss → backward → step

torch.manual_seed(0)
N, D, H, C = 8, 4, 8, 3  # batch、輸入維度、隱藏層、類別數
x = torch.randn(N, D)    # 輸入不需要 requires_grad；梯度不要流回資料
y = torch.randint(0, C, (N,))

# Linear 的 weight、bias 建立時就 requires_grad=True，會進 model.parameters()
model = torch.nn.Sequential(
    torch.nn.Linear(D, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, C),
)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
CEL = torch.nn.CrossEntropyLoss()  # 吃最後一層的 logits，內部做 log_softmax

w_before = model[0].weight.detach().clone()

for step in range(20):
    opt.zero_grad()          # 清掉上一步殘留的 .grad，否則會累加
    logits = model(x)        # forward，形狀 (N, C)
    loss = CEL(logits, y)
    loss.backward()          # 梯度寫進各層 weight.grad / bias.grad；權重數值還沒變
    opt.step()               # param -= lr * grad，這時權重才更新
    if step == 0 or step == 19:
        print(f"step {step:02d}  loss={loss.item():.4f}")

print("第一層 weight 有沒有變", not torch.equal(w_before, model[0].weight.detach()))
print("x.grad 應是 None  ", x.grad)

step 00  loss=1.2518
step 19  loss=0.9264
第一層 weight 有沒有變 True
x.grad 應是 None   None


In [5]:
# 四種寫法，同一組 x。差在 forward 怎麼接、參數怎麼註冊。
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
x = torch.randn(2, 4)  # (N, D)
D, H, C = 4, 8, 3
# 1. Sequential：直線，上一層輸出就是下一層輸入。不能分支、殘差。
seq = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, C))
print("1 Sequential ", tuple(seq(x).shape))

1 Sequential  (2, 3)


In [6]:

# 2. nn.Module：forward 自己接。
#    有參數的用 nn.Linear（self.xxx = Module 才進 parameters()）
#    沒參數的用 F.relu，它只是函數，不算一層。
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(D, H)
        self.fc2 = nn.Linear(H, C)
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        return self.fc2(x)
print("2 Module+F ", tuple(MLP()(x).shape))

2 Module+F  (2, 3)


In [7]:

# 3. ModuleList：層數不固定。普通 Python list 裝 Linear 
#   不會註冊，optimizer 會漏掉。
class Deep(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(D, H),
            nn.Linear(H, H),
            nn.Linear(H, C),
        ])
    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return x
print("3 ModuleList", tuple(Deep()(x).shape))
# ModuleDict：按名字取層，一樣會註冊參數。
class Named(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleDict({
            "in": nn.Linear(D, H),
            "out": nn.Linear(H, C),
        })
    def forward(self, x):
        return self.blocks["out"](F.relu(self.blocks["in"](x)))
print("3 ModuleDict", tuple(Named()(x).shape))

3 ModuleList (2, 3)
3 ModuleDict (2, 3)


In [8]:

# 4. 純 functional：權重自己用 nn.Parameter，運算全走 F.linear / 
# F.relu / F.cross_entropy。
#    F.linear 的 weight 形狀是 (out, in)，也就是 y = x @ W.T + b。
class FunctionalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.w1 = nn.Parameter(torch.empty(H, D))
        self.b1 = nn.Parameter(torch.zeros(H))
        self.w2 = nn.Parameter(torch.empty(C, H))
        self.b2 = nn.Parameter(torch.zeros(C))
        nn.init.kaiming_uniform_(self.w1, a=5**0.5)
        nn.init.kaiming_uniform_(self.w2, a=5**0.5)
    def forward(self, x):
        x = F.linear(x, self.w1, self.b1)
        x = F.relu(x)
        return F.linear(x, self.w2, self.b2)
fm = FunctionalMLP()
logits = fm(x)
y = torch.tensor([0, 2])
loss = F.cross_entropy(logits, y)  # 等價 
nn.CrossEntropyLoss()(logits, y)
loss.backward()
print("4 functional", tuple(logits.shape), "loss", 
round(loss.item(), 4), "w1.grad", tuple(fm.w1.grad.shape))

4 functional (2, 3) loss 1.2951 w1.grad (8, 4)


In [14]:
# class 寫法的完整訓練
# 被更新的是 fc1、fc2 的 weight / bias，不是 x，也不是 logits。
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, d, h, c):
        super().__init__()          # 先建好 Module 的參數登記表
        self.fc1 = nn.Linear(d, h)  # 有 weight、bias，會進 # parameters()
        self.fc2 = nn.Linear(h, c)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))     # relu 沒有參數
        return self.fc2(x)          # 回傳 logits (N, C)，還沒做 # softmax
    
torch.manual_seed(0)

N, D, H, C = 8, 4, 8, 3
x = torch.randn(N, D)              # 輸入資料，不要 requires_grad
y = torch.randint(0, C, (N,))      # 類別編號 (N,)

model = MLP(D, H, C)

opt = torch.optim.SGD(model.parameters(), lr=0.1)  # # 只會更新登記過的參數

w_before = model.fc1.weight.detach().clone()

for step in range(20):
    opt.zero_grad()                        # 清掉上一步的 .grad
    logits = model(x)                      # forward
    loss = F.cross_entropy(logits, y)      # 吃 logits，內部做 # log_softmax
    loss.backward()                        # 梯度寫進 # fc1.weight.grad 等，數值還沒變
    opt.step()                             # param -= lr * # grad，這時權重才更新
    if step == 0 or step == 19:
        print(f"step {step:02d}  loss={loss.item():.4f}")

print("fc1.weight 有變", not torch.equal(w_before, 

model.fc1.weight.detach()))

print("x.grad", x.grad)            # None：x 不是參數


step 00  loss=1.2518
step 19  loss=0.9264
fc1.weight 有變 True
x.grad None


In [ ]:
# Dataset 負責「一筆資料」；DataLoader 負責「把它們組成 batch」。
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def make_xy(n, d, seed):
    # 類別由 x 的前兩維決定，所以模型學得會。亂貼標籤的話，val 準確率不會上升。
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, d, generator=g)
    score = x[:, 0] + 0.5 * x[:, 1]
    y = torch.bucketize(score, torch.tensor([-0.5, 0.5]))  # 0、1、2
    return x, y

class ToySet(Dataset):
    def __init__(self, n, d, seed):
        self.x, self.y = make_xy(n, d, seed)

    def __len__(self):
        return self.x.shape[0]       # 一共幾筆

    def __getitem__(self, i):
        return self.x[i], self.y[i]  # 第 i 筆：(特徵, 類別編號)

# seed 不同：兩批不同的 x，規則相同。val 不參與更新，只拿來量準確率。
train_ds = ToySet(64, 4, seed=0)
val_ds = ToySet(64, 4, seed=1)

# x、y 已經是 tensor 時，不必寫 class，用 TensorDataset(x, y) 即可。
# shuffle：每個 epoch 打亂訓練順序。val 不要 shuffle，每次 eval 的順序才固定。
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

xb, yb = next(iter(train_loader))
print("資料筆數", len(train_ds), "一個 batch", tuple(xb.shape), tuple(yb.shape), "batch 數", len(train_loader))


In [ ]:
# 用上一格的 train_loader / val_loader。
# 一步還是 zero_grad → forward → loss → backward → step，只是現在對每個 batch 做一次。

class MLP(nn.Module):
    def __init__(self, d, h, c):
        super().__init__()
        self.fc1 = nn.Linear(d, h)
        self.fc2 = nn.Linear(h, c)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

@torch.no_grad()
def accuracy(model, loader):
    """固定資料上的 eval：預測類別答對的比例。不算梯度，也不更新權重。"""
    model.eval()  # Dropout / BatchNorm 改走推論。這個 MLP 沒有這兩層，數值不會因此改變。
    correct = total = 0
    for x, y in loader:
        pred = model(x).argmax(dim=-1)  # 每個樣本取 logits 最大的類別
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

torch.manual_seed(0)
model = MLP(4, 16, 3)
opt = torch.optim.SGD(model.parameters(), lr=0.2)

print("訓練前 val acc", round(accuracy(model, val_loader), 3))

model.train()  # 回到訓練行為再更新
for epoch in range(40):
    for x, y in train_loader:
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()

print("訓練後 train acc", round(accuracy(model, train_loader), 3))
print("訓練後 val acc  ", round(accuracy(model, val_loader), 3))
